# CSE 164 A100 Supervised Multi-Task Runs

This notebook is for Google Colab A100 runs. It keeps the same Drive data layout as the existing Colab notebook:

```text
/content/drive/MyDrive/CSE164FinalProject/raw/
|-- metadata/
|-- test/
|-- train_labeled/
|-- train_seg/
|-- train_unlabeled/
`-- val/
```

Outputs are written to `/content/drive/MyDrive/CSE164FinalProject/outputs/` so checkpoints and submissions survive runtime resets.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Wed Jun 10 23:46:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Dataset set up

In [10]:
!mkdir -p /content/data

!cp /content/drive/MyDrive/CSE164FinalProject/raw.zip /content/raw.zip

!unzip -q /content/raw.zip -d /content/data

!ls /content/data
!ls /content/data/raw

raw
metadata  test	train_labeled  train_seg  train_unlabeled  val


# Clone or update Repo

In [4]:
import getpass

token = getpass.getpass("GitHub Token: ")

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone https://{token}@github.com/007deepaks/CSE-164FinalProject.git
    %cd {REPO_DIR}

!git status --short

GitHub Token: ··········
/content
Cloning into 'CSE-164FinalProject'...
remote: Enumerating objects: 288, done.
remote: Counting objects: 100% (288/288), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 288 (delta 175), reused 192 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (288/288), 2.39 MiB | 1.95 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/CSE-164FinalProject


## Repository and Paths

If the repo is not already in `/content/CSE-164FinalProject`, upload it, clone it, or copy it from Drive before continuing.

In [11]:
from pathlib import Path
import os

REPO_DIR = Path('/content/CSE-164FinalProject')

# local SSD (fast)
DATA_ROOT = Path('/content/data/raw')

# checkpoints still saved to Drive
DRIVE_PROJECT = Path('/content/drive/MyDrive/CSE164FinalProject')
DRIVE_OUTPUTS = DRIVE_PROJECT / 'outputs'

assert REPO_DIR.exists(), f'Missing repo at {REPO_DIR}'
assert DATA_ROOT.exists(), f'Missing data at {DATA_ROOT}'

DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)

print('repo:', REPO_DIR)
print('data:', DATA_ROOT)
print('drive outputs:', DRIVE_OUTPUTS)

repo: /content/CSE-164FinalProject
data: /content/data/raw
drive outputs: /content/drive/MyDrive/CSE164FinalProject/outputs


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q kaggle

In [7]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import torch
print(torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
2.11.0+cu128
cuda available: True
NVIDIA A100-SXM4-40GB


## Smoke Test

Runs one tiny ResNet-50 multi-task epoch to verify the notebook, data paths, checkpoint writing, and validation reload path.

In [12]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnet50 \
  --stage joint \
  --epochs 1 \
  --warmup-epochs 0 \
  --image-size 128 \
  --num-segmentation-classes 1 \
  --seg-batch-size 1 \
  --cls-batch-size 2 \
  --val-batch-size 1 \
  --num-workers 2 \
  --learning-rate 1e-4 \
  --min-learning-rate 1e-6 \
  --weight-decay 1e-4 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 1.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --max-seg-samples 2 \
  --max-cls-samples 4 \
  --max-val-samples 2 \
  --no-random-crop \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnet50_smoke"

Using device: cuda; mixed precision: True
Multi-task ResNet-50 from scratch: image_size=128, decoder=unet, num_segmentation_classes=1, classifier_dropout=0.2
Train samples: segmentation=2, classification=4; val=2
LR scheduler: cosine decay from 0.0001 to 1e-06
Training stage: joint

Epoch 1/1
/content/CSE-164FinalProject/src/training/train_multitask.py:811: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
  seg_loss=0.0000 cls_loss=0.0000 train_cls_acc=0.0000 train_seg_cls_acc=0.0000 val_auto=0.0015 val_seg=0.0022 mIoU=0.0000 bin_iou=0.0077 oracle_mIoU=0.0077 boundary=0.0109 rare_mIoU=0.0000 macro_acc=0.0000 thr=0.50 lr=0.000001
  s

## Serious ResNet-50 Supervised Run

This trains ResNet-50 from scratch with shared encoder, U-Net binary decoder, mask-guided classifier pooling, EMA, warmup plus cosine LR, and weighted supervised sampling.

In [9]:
!python -m src.training.train_multitask \
  --data-root "$DATA_ROOT" \
  --architecture resnet50 \
  --stage joint \
  --epochs 100 \
  --warmup-epochs 3 \
  --image-size 384 \
  --num-segmentation-classes 1 \
  --seg-batch-size 12 \
  --cls-batch-size 96 \
  --val-batch-size 12 \
  --num-workers 8 \
  --learning-rate 1e-3 \
  --min-learning-rate 1e-6 \
  --weight-decay 5e-2 \
  --label-smoothing 0.1 \
  --segmentation-loss-weight 1.0 \
  --dice-loss-weight 1.0 \
  --seg-classification-loss-weight 1.0 \
  --cls-loss-weight 1.0 \
  --gradient-clip 5.0 \
  --ema-decay 0.9998 \
  --weighted-combined-sampling \
  --class-only-sample-weight 1.0 \
  --mask-sample-weight 2.5 \
  --validation-threshold 0.50 \
  --checkpoint-dir "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384"

Using device: cuda; mixed precision: True
Multi-task ResNet-50 from scratch: image_size=384, decoder=unet, num_segmentation_classes=1, classifier_dropout=0.2
Train samples: segmentation=3000, classification=10500; val=750
LR scheduler: linear warmup for 3 epochs to 0.001, then cosine decay to 1e-06
Training stage: joint

Epoch 1/100
  mixed batch 0020/750 seg_loss=7.4052 cls_loss=6.2998
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 912, in <module>
    main()
  File "/content/CSE-164FinalProject/src/training/train_multitask.py", line 889, in main
    run_one_stage(args)
object address  : 0x7cb0b934c220
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


## Threshold Sweep

Run after training finishes. Pick the threshold with the best automated validation score.

In [ ]:
!python -m src.training.tune_multitask_threshold \
  --seg-checkpoint "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 8 \
  --tta hflip \
  --thresholds 0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75

## Generate Submission

Replace `BEST_THRESHOLD` with the best threshold from the sweep.

In [ ]:
BEST_THRESHOLD = 0.50

In [ ]:
!python -m src.training.predict_multitask \
  --checkpoint "$DRIVE_OUTPUTS/checkpoints/resnet50_multitask_384/best_multitask.pt" \
  --data-root "$DATA_ROOT" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 8 \
  --tta hflip \
  --seg-threshold "$BEST_THRESHOLD" \
  --output "$DRIVE_OUTPUTS/submissions/resnet50_multitask_384_tta.csv"

In [ ]:
!python starter/validate_submission_csv.py \
  --submission "$DRIVE_OUTPUTS/submissions/resnet50_multitask_384_tta.csv" \
  --data-root "$DATA_ROOT" \
  --split test